# 01 · Explore

Confirm the corpus is balanced enough to train on, and that each fault shows the signature ADR-0002 predicted. If a prediction fails here, ADR-0002 gets a correction — not this notebook.

In [ ]:
from pathlib import Path

import pandas as pd

import features

META = ["window_start_ns", "window_end_ns", "label", "fault_kind", "severity", "command_id"]
PARQUET = Path("..") / "data" / f"train-{features.__version__}.parquet"

df = pd.read_parquet(PARQUET)
feature_cols = [c for c in df.columns if c not in META]
X = df[feature_cols].to_numpy(dtype=float)
kinds = df["fault_kind"].tolist()
starts = df["window_start_ns"].to_numpy()
print(f"{len(df)} windows, {len(feature_cols)} features, features v{features.__version__}")

## Class balance

In [ ]:
print(df["fault_kind"].value_counts())

## Signature check (ADR-0002)

Expected: encoder → elbow `pos_std` up; stuck & friction → elbow `vel_rms` toward zero; dropout → `n_samples` down.

In [ ]:
sig = ["elbow_joint__vel_rms", "elbow_joint__pos_std", "n_samples"]
means = df.groupby("fault_kind")[sig].mean()
print(means.round(3))

In [ ]:
normal = means.loc["none"]
assert means.loc["stuck", "elbow_joint__vel_rms"] < normal["elbow_joint__vel_rms"]
assert means.loc["friction", "elbow_joint__vel_rms"] < normal["elbow_joint__vel_rms"]
assert means.loc["encoder", "elbow_joint__pos_std"] > normal["elbow_joint__pos_std"]
assert means.loc["dropout", "n_samples"] < normal["n_samples"]
print("ADR-0002 signatures confirmed on this corpus")